# Telco Customer Churn EDA\n\nThis notebook analyzes customer churn patterns and revenue loss using the Telco Customer Churn dataset.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nfrom pathlib import Path\n\nDATA_PATH = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')\ndf = pd.read_csv(DATA_PATH)\ndf.head()

In [ ]:
df.info()\ndf.describe(include='all')

## 1. Data Cleaning\n\n`TotalCharges` contains blank values, so it should be converted to numeric with invalid values treated as missing.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')\ndf['ChurnFlag'] = df['Churn'].map({'Yes': 1, 'No': 0})\n\ndf['tenure_group'] = pd.cut(\n    df['tenure'],\n    bins=[-1, 6, 12, 24, 48, 999],\n    labels=['0-6 months', '7-12 months', '13-24 months', '25-48 months', '49+ months']\n)\n\ndf.isna().sum()

## 2. Overall Churn and Revenue Loss

In [ ]:
total_customers = len(df)\nchurned_customers = df['ChurnFlag'].sum()\nchurn_rate = df['ChurnFlag'].mean() * 100\ntotal_monthly_revenue = df['MonthlyCharges'].sum()\nchurned_monthly_revenue = df.loc[df['Churn'] == 'Yes', 'MonthlyCharges'].sum()\nrevenue_loss_rate = churned_monthly_revenue / total_monthly_revenue * 100\n\nsummary = pd.DataFrame({\n    'metric': [\n        'total_customers',\n        'churned_customers',\n        'churn_rate',\n        'total_monthly_revenue',\n        'churned_monthly_revenue',\n        'revenue_loss_rate'\n    ],\n    'value': [\n        total_customers,\n        churned_customers,\n        round(churn_rate, 2),\n        round(total_monthly_revenue, 2),\n        round(churned_monthly_revenue, 2),\n        round(revenue_loss_rate, 2)\n    ]\n})\nsummary

## 3. Group-level Churn Analysis

In [ ]:
def churn_summary(group_col):\n    result = (\n        df.groupby(group_col)\n        .agg(\n            total_customers=('customerID', 'count'),\n            churned_customers=('ChurnFlag', 'sum'),\n            churn_rate=('ChurnFlag', lambda x: round(x.mean() * 100, 2)),\n            avg_monthly_charges=('MonthlyCharges', lambda x: round(x.mean(), 2)),\n            churned_monthly_revenue=('MonthlyCharges', lambda x: round(x[df.loc[x.index, 'Churn'] == 'Yes'].sum(), 2))\n        )\n        .reset_index()\n        .sort_values('churn_rate', ascending=False)\n    )\n    return result\n\ncontract_summary = churn_summary('Contract')\npayment_summary = churn_summary('PaymentMethod')\ninternet_summary = churn_summary('InternetService')\ntenure_summary = churn_summary('tenure_group')\n\ncontract_summary

In [ ]:
payment_summary

In [ ]:
internet_summary

In [ ]:
tenure_summary

## 4. Visualization

In [ ]:
Path('../images').mkdir(exist_ok=True)\n\ndef plot_churn_rate(summary_df, x_col, title, filename):\n    ax = summary_df.plot(kind='bar', x=x_col, y='churn_rate', legend=False, figsize=(8, 5))\n    ax.set_title(title)\n    ax.set_ylabel('Churn Rate (%)')\n    ax.set_xlabel(x_col)\n    plt.xticks(rotation=30, ha='right')\n    plt.tight_layout()\n    plt.savefig(f'../images/{filename}', dpi=150)\n    plt.show()\n\nplot_churn_rate(contract_summary, 'Contract', 'Churn Rate by Contract Type', 'churn_rate_by_contract.png')

In [ ]:
plot_churn_rate(payment_summary, 'PaymentMethod', 'Churn Rate by Payment Method', 'churn_rate_by_payment_method.png')

In [ ]:
plot_churn_rate(internet_summary, 'InternetService', 'Churn Rate by Internet Service', 'churn_rate_by_internet_service.png')

In [ ]:
plot_churn_rate(tenure_summary, 'tenure_group', 'Churn Rate by Tenure Group', 'churn_rate_by_tenure_group.png')

## 5. Business Interpretation\n\n- Churned customers account for a meaningful portion of monthly charges.\n- Month-to-month customers should be prioritized for retention.\n- Early-tenure customers need stronger onboarding.\n- Retention campaigns should target high-risk and high-revenue segments first.